In [2]:
import pandas as pd
import numpy as np

np.random.seed(3)

# ---- 1. Cargar datos (SIN encabezado, por eso header=None) ----
columnas = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df = pd.read_csv('iris.data', header=None, names=columnas)

# ---- 2. Quedarnos solo con 2 clases (Setosa y Versicolor) ----
df_binario = df[df['species'].isin(['Iris-setosa', 'Iris-versicolor'])].reset_index(drop=True)

# ---- 3. Separar features (X) y target (D) ----
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X_raw = df_binario[features].values
# Target binario: Setosa = 0, Versicolor = 1
D_raw = (df_binario['species'] == 'Iris-versicolor').astype(int).values.reshape(-1, 1)

# ---- 4. Normalizar ----
mean = X_raw.mean(axis=0)
std = X_raw.std(axis=0)
X_norm = (X_raw - mean) / std

# ---- 5. Split train/test ----
np.random.seed(42)
idx = np.random.permutation(len(X_norm))
split = int(0.8 * len(X_norm))
train_idx, test_idx = idx[:split], idx[split:]
X_train, D_train = X_norm[train_idx], D_raw[train_idx]
X_test, D_test = X_norm[test_idx], D_raw[test_idx]

# ---- 6. Activación ----
def sigmoid(v):
    return 1 / (1 + np.exp(-v))

# ---- 7. Pesos iniciales ----
n_hidden = 4
W1 = 2 * np.random.rand(n_hidden, 4) - 1
W2 = 2 * np.random.rand(1, n_hidden) - 1
alpha = 0.5

# ---- 8. Entrenamiento ----
def train_step(W1, W2, X, D):
    for k in range(len(X)):
        x = X[k].reshape(-1, 1)
        d = D[k].reshape(-1, 1)

        v1 = W1 @ x
        y1 = sigmoid(v1)
        v = W2 @ y1
        y = sigmoid(v)

        e = d - y
        delta = y * (1 - y) * e

        e1 = W2.T @ delta
        delta1 = y1 * (1 - y1) * e1

        W1 = W1 + alpha * (delta1 @ x.T)
        W2 = W2 + alpha * (delta @ y1.T)
    return W1, W2

for epoch in range(3000):
    W1, W2 = train_step(W1, W2, X_train, D_train)

# ---- 9. Evaluar en test ----
correctas = 0
for k in range(len(X_test)):
    x = X_test[k].reshape(-1, 1)
    v1 = W1 @ x
    y1 = sigmoid(v1)
    v = W2 @ y1
    y = sigmoid(v)
    pred = 1 if y[0,0] > 0.5 else 0
    if pred == D_test[k, 0]:
        correctas += 1

print(f"Accuracy en test: {correctas/len(X_test)*100:.1f}%")

Accuracy en test: 100.0%


In [3]:
import pandas as pd
import numpy as np

np.random.seed(3)

# ---- 1. Cargar TODAS las clases (ya no filtramos) ----
columnas = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df = pd.read_csv('iris.data', header=None, names=columnas)

# ---- 2. Features y target ----
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X_raw = df[features].values
clases = sorted(df['species'].unique())  # ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']

# One-hot: cada fila tiene un 1 en la posición de su especie
D_raw = np.zeros((len(df), 3))
for i, especie in enumerate(df['species'].values):
    D_raw[i, clases.index(especie)] = 1

# ---- 3. Normalizar ----
mean = X_raw.mean(axis=0)
std = X_raw.std(axis=0)
X_norm = (X_raw - mean) / std

# ---- 4. Split ----
np.random.seed(42)
idx = np.random.permutation(len(X_norm))
split = int(0.8 * len(X_norm))
train_idx, test_idx = idx[:split], idx[split:]
X_train, D_train = X_norm[train_idx], D_raw[train_idx]
X_test, D_test = X_norm[test_idx], D_raw[test_idx]

# ---- 5. Activaciones ----
def sigmoid(v):
    return 1 / (1 + np.exp(-v))

def softmax(v):
    ex = np.exp(v - np.max(v))
    return ex / np.sum(ex)

# ---- 6. Pesos ----
n_hidden = 8
W1 = 2 * np.random.rand(n_hidden, 4) - 1
W2 = 2 * np.random.rand(3, n_hidden) - 1   # 3 salidas ahora, no 1
alpha = 0.5

# ---- 7. Entrenamiento ----
def train_step(W1, W2, X, D):
    for k in range(len(X)):
        x = X[k].reshape(-1, 1)
        d = D[k].reshape(-1, 1)

        v1 = W1 @ x
        y1 = sigmoid(v1)
        v = W2 @ y1
        y = softmax(v)          # softmax en vez de sigmoide en la salida

        e = d - y
        delta = e                # se simplifica, igual que en el ejemplo de dígitos

        e1 = W2.T @ delta
        delta1 = y1 * (1 - y1) * e1

        W1 = W1 + alpha * (delta1 @ x.T)
        W2 = W2 + alpha * (delta @ y1.T)
    return W1, W2

for epoch in range(3000):
    W1, W2 = train_step(W1, W2, X_train, D_train)

# ---- 8. Evaluar en test, guardando también los errores ----
correctas = 0
errores = []
for k in range(len(X_test)):
    x = X_test[k].reshape(-1, 1)
    v1 = W1 @ x
    y1 = sigmoid(v1)
    v = W2 @ y1
    y = softmax(v)

    pred = np.argmax(y)
    real = np.argmax(D_test[k])

    if pred == real:
        correctas += 1
    else:
        errores.append((clases[real], clases[pred], y.ravel().round(3)))

print(f"Accuracy en test: {correctas/len(X_test)*100:.1f}%  ({correctas}/{len(X_test)})")
print("\nErrores (real -> predicho, probabilidades):")
for real, pred, probs in errores:
    print(f"  {real} -> {pred}  {dict(zip(clases, probs))}")

Accuracy en test: 93.3%  (28/30)

Errores (real -> predicho, probabilidades):
  Iris-virginica -> Iris-versicolor  {'Iris-setosa': np.float64(0.0), 'Iris-versicolor': np.float64(1.0), 'Iris-virginica': np.float64(0.0)}
  Iris-virginica -> Iris-versicolor  {'Iris-setosa': np.float64(0.0), 'Iris-versicolor': np.float64(0.97), 'Iris-virginica': np.float64(0.03)}
